In [ ]:
import sqlite3

DB_NAME = "Nachname_Vorname_fitnessstudio.db"

def create_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def create_tables(conn):
    cursor = conn.cursor()

    # Vorhandene Tabellen löschen, damit das Skript mehrfach ausführbar ist
    cursor.execute("DROP TABLE IF EXISTS anmeldung;")
    cursor.execute("DROP TABLE IF EXISTS kurse;")
    cursor.execute("DROP TABLE IF EXISTS mitglied;")
    cursor.execute("DROP TABLE IF EXISTS trainer;")
    cursor.execute("DROP TABLE IF EXISTS fitnessstudio;")

    # Tabelle Fitnessstudio
    cursor.execute("""
        CREATE TABLE fitnessstudio (
            fitness_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL
        );
    """)

    # Tabelle Trainer
    cursor.execute("""
        CREATE TABLE trainer (
            trainer_id INTEGER PRIMARY KEY AUTOINCREMENT,
            vorname TEXT NOT NULL,
            nachname TEXT NOT NULL,
            spezialgebiet TEXT
        );
    """)

    # Tabelle Mitglied
    cursor.execute("""
        CREATE TABLE mitglied (
            mitglied_id INTEGER PRIMARY KEY AUTOINCREMENT,
            vorname TEXT NOT NULL,
            nachname TEXT NOT NULL,
            email TEXT NOT NULL UNIQUE,
            beitrittsdatum DATE NOT NULL
        );
    """)

    # Tabelle Kurse
    cursor.execute("""
        CREATE TABLE kurse (
            kurs_id INTEGER PRIMARY KEY AUTOINCREMENT,
            fitness_id INTEGER NOT NULL,
            trainer_id INTEGER NOT NULL,
            wochentag TEXT NOT NULL,
            bezeichnung TEXT NOT NULL,
            uhrzeit TEXT NOT NULL,
            max_teilnehmer INTEGER NOT NULL,
            FOREIGN KEY (fitness_id) REFERENCES fitnessstudio(fitness_id),
            FOREIGN KEY (trainer_id) REFERENCES trainer(trainer_id)
        );
    """)

    # Tabelle Anmeldung
    cursor.execute("""
        CREATE TABLE anmeldung (
            anmelde_id INTEGER PRIMARY KEY AUTOINCREMENT,
            kurs_id INTEGER NOT NULL,
            mitglied_id INTEGER NOT NULL,
            FOREIGN KEY (kurs_id) REFERENCES kurse(kurs_id),
            FOREIGN KEY (mitglied_id) REFERENCES mitglied(mitglied_id)
        );
    """)

    conn.commit()

def insert_sample_data(conn):
    cursor = conn.cursor()

    # 1 Fitnessstudio
    cursor.execute("""
        INSERT INTO fitnessstudio (name)
        VALUES (?)
    """, ("FitLife Studio",))

    # Mindestens 3 Trainer
    trainer_daten = [
        ("Anna", "Huber", "Yoga"),
        ("Markus", "Leitner", "Krafttraining"),
        ("Sophie", "Gruber", "Cardio"),
    ]
    cursor.executemany("""
        INSERT INTO trainer (vorname, nachname, spezialgebiet)
        VALUES (?, ?, ?)
    """, trainer_daten)

    # Mindestens 6 Mitglieder
    mitglied_daten = [
        ("Lena", "Maier", "lena.maier@example.com", "2024-01-15"),
        ("Paul", "Berger", "paul.berger@example.com", "2024-02-10"),
        ("Mia", "Schwarz", "mia.schwarz@example.com", "2024-03-05"),
        ("Jonas", "Hofer", "jonas.hofer@example.com", "2024-04-20"),
        ("Emma", "Winkler", "emma.winkler@example.com", "2024-05-12"),
        ("Noah", "Fuchs", "noah.fuchs@example.com", "2024-06-01"),
    ]
    cursor.executemany("""
        INSERT INTO mitglied (vorname, nachname, email, beitrittsdatum)
        VALUES (?, ?, ?, ?)
    """, mitglied_daten)

    # Mindestens 5 Kurse
    kurs_daten = [
        (1, 1, "Montag", "Yoga Basic", "2025-05-12 09:00:00", 15),
        (1, 2, "Dienstag", "Power Workout", "2025-05-13 18:00:00", 12),
        (1, 3, "Mittwoch", "Cardio Blast", "2025-05-14 17:30:00", 20),
        (1, 1, "Donnerstag", "Stretch & Relax", "2025-05-15 10:00:00", 10),
        (1, 2, "Freitag", "Muskelaufbau", "2025-05-16 16:00:00", 14),
    ]
    cursor.executemany("""
        INSERT INTO kurse (fitness_id, trainer_id, wochentag, bezeichnung, uhrzeit, max_teilnehmer)
        VALUES (?, ?, ?, ?, ?, ?)
    """, kurs_daten)

    # Mindestens 8 Anmeldungen
    anmeldungen = [
        (1, 1),
        (1, 2),
        (2, 3),
        (2, 4),
        (3, 5),
        (3, 6),
        (4, 1),
        (5, 2),
        (5, 3),
    ]
    cursor.executemany("""
        INSERT INTO anmeldung (kurs_id, mitglied_id)
        VALUES (?, ?)
    """, anmeldungen)

    conn.commit()

def show_data(conn):
    cursor = conn.cursor()

    print("\n--- FITNESSSTUDIO ---")
    for row in cursor.execute("SELECT * FROM fitnessstudio;"):
        print(row)

    print("\n--- TRAINER ---")
    for row in cursor.execute("SELECT * FROM trainer;"):
        print(row)

    print("\n--- MITGLIEDER ---")
    for row in cursor.execute("SELECT * FROM mitglied;"):
        print(row)

    print("\n--- KURSE ---")
    for row in cursor.execute("SELECT * FROM kurse;"):
        print(row)

    print("\n--- ANMELDUNGEN ---")
    for row in cursor.execute("SELECT * FROM anmeldung;"):
        print(row)

def main():
    conn = create_connection()
    try:
        create_tables(conn)
        insert_sample_data(conn)
        show_data(conn)
        print(f"\nDatenbank '{DB_NAME}' wurde erfolgreich erstellt und befüllt.")
    except sqlite3.Error as e:
        print("Fehler bei der Arbeit mit SQLite:", e)
    finally:
        conn.close()

if __name__ == "__main__":
    main()